# A single-band model for La$_{2}$CuO$_{4}$: Post-process

In the previous tutorial, you solved an effective one-band model for La$_{2}$CuO$_{4}$ derived from first-principles within DMFT. We will now learn how to analyze and post-process the results we obtained in the [previous tutorial](). In this tutorial, you will learn how to:

- Analytically continue Matsubara Green's functions and self-energies
- Post-process and interpret spectral functions and self-energies

### 🧪 Exercise 0: Load the converged DMFT from HDF5

### 🧪 Exercise 1: Estimate Z from the Matsubara self-energy

Using the self-energy that you obtained above, estimate the quasiparticle renormalization factor and the scattering rate, which are defined as 

$$ Z = (1 - \partial_{\omega_{n}}\text{Im}\Sigma(i\omega_{n}) |_{\omega_{n}\rightarrow 0})^{-1}.$$

We can estimate this quantity a couple of different ways. The simplest is to just look at the first Matsubara point,

$$ Z \sim \Big (1 - \frac{\mathrm{Im}\Sigma(i\omega_{0})}{\omega_{0}} \Big )^{-1},$$
where $\omega_{0} = \frac{\pi}{\beta}$.

## 🌀 Analytic continuation: from Matsubara to real frequencies

In this section, we will learn how to **analytically continue** our results from Matsubara (imaginary) frequencies onto the real-frequency axis, so that we may plot spectral functions. Because analytic continuation is an ill-posed problem, it is notoriously unstable, and the method itself remains an active area of research. However, there are two common analytic continuation techniques: Padé analytic continuation and the maximum entropy (MaxEnt) method. In this section, we will use Padé analytic contintuation since it is simpler. There is an additional tutorial using the MaxEnt method to continue the data.

### ❓The problem
We assume the Matsubra Green's function $G(i\omega_{n})$ corresponds to a dynamic correlation function $G(\omega)$ where $\omega_{n}$ is the Matsubara frequency and $\omega$ is the real frequency. The function $G(z)$ is analytic in the whole upper complex plane of $z$ excluding the real axis, the retarded Green function can be obtained by analytic contintuation $$G(\omega) = G(i\omega_{n}\rightarrow \omega + i0^{+})$$. The spectral function $A(\omega)$ is determined by taking the imaginary part of $G(\omega + i0^{+})$ by the formula

$$ A(\omega) = -\frac{1}{\pi}\mathrm{Im}G(\omega + i0^{+})$$

### 📐 Padé method
Padé analytic continuation is based on the assumption that the Matsubara Green function can be approximated by a rational function of degree $r$

$$ G(z) = \frac{\sum_{i=0}^{r-1} p_{i}z^{i}}{\sum_{j=0}^{r-1}q_{j}z^{j} + z^{r}}, $$
where $p_{i}$ and $q_{j}$ are complex. The coefficients can be determined by solving $2r$ linear equatiions from $2r$ arbitrary but different input points ${i\omega_{n}, G(i\omega_{n})}$. Once the coefficients are determined, we replace $z$ with $\omega + i0^{+}$ to obtain the retarded correlation function.

### 🧪 Exercise 2: Analytically continue the Matsubara Green's function

In this exercise, we will analytically continue the Matsubara Green's function onto the real frequency axis.

#### 🧩 Your task
1. set up a ``BlockGf`` with a real frequency mesh ``MeshReFreq``.
2. For each block in the Matsubara Green's function, use the ``set_from_pade`` method to obtain the real frequency Green's function for each block.
4. Plot the resulting spectral function using the formula $A(\omega) = -\frac{1}{\pi}\text{Im}G(\omega + i0^{+})$.

In [ ]:
from triqs.gf import BlockGf, MeshReFreq

n_w = 1000
window = (-10, 10)

# Step 1: set up real-frequency mesh and BlockGf

## get the block shape of the Matsubara Green's function
gf_struct = [ (bl, g.target_shape[0]) for bl, g in solver_results.G_iw ]

## setup real-frequency mesh
mesh = MeshReFreq(window, n_w)

## construct Block Green's function on real-frequency mesh
G_w = BlockGf(mesh=mesh, gf_struct=gf_struct)

# Step 2: loop over the blocks and call set_from_pade
n_points = 100
freq_offset = 0.0
for block, g in G_w: 
    g.set_from_pade(solver_results.G_iw[block], n_points=n_points, freq_offset=freq_offset)

# Step 3: plot the spectral function
fig, ax = plt.subplots(figsize=(5,3))
ax.oplot(-1/np.pi*G_w.imag, lw=1)
ax.set_ylabel(r'$A(\omega)$'); ax.set_xlim(-5,5); ax.set_ylim(0, 1)
plt.show()

> ⚠️ **Warning**: The Pade method can be sensitive to the choice of ``n_points``, which correpsonds to the amount of Matsubara data you allow the algorithm to see. Try changing this parameter to see how the resulting spectral function changes.

In [ ]:
from utils.maxent import Aw_from_maxent

mesh, Aw = Aw_from_maxent(solver_results.G_iw, omega_min=-10, omega_max=10, alpha_min=1e-6, alpha_max=1e2, n_alpha_points=50, n_omega_points=200, error=0.04)

In [ ]:
fig, ax = plt.subplots(figsize=(5,3))
ax.plot(mesh, Aw['up_0'].get_A_out('LineFitAnalyzer')[0,0].real, lw=1, label='MaxEnt')
ax.oplot(-1/np.pi*G_w['up_0'].imag, lw=1, label='Pade')
ax.set_ylabel(r'$A(\omega)$'); ax.set_xlim(-5,5); ax.set_ylim(0, 1)
plt.show()

### 🧪 Exercise 3: Understanding your spectral function

Let's take a closer look out the spectral functions that we just obtained. Specifically, let's see if it reproduces the features that we expect, which are spectral weight transfer into the upper and lower Hubbard bands around $\pm U/2$ as well as the width of the quasiparticle peak which goes as $\sim Z \Gamma$, where $\Gamma = -\mathrm{Im}\Sigma(0)$.

In [ ]:
fig, ax = plt.subplots(figsize=(5,3))
ax.plot(mesh, Aw['up_0'].get_A_out('LineFitAnalyzer')[0,0].real, lw=1, label='MaxEnt')
ax.oplot(-1/np.pi*G_w['up_0'].imag, lw=1, label='Pade')
ax.set_ylabel(r'$A(\omega)$'); ax.set_xlim(-5,5); ax.set_ylim(0, 1)
ax.legend(loc='lower left')
# We expect upper and lower Hubbard bands at +/- U/2
ax.axvline(+U/2, color='grey', lw=4, alpha=0.25); ax.axvline(-U/2, color='grey', lw=4, alpha=0.25)
ax.text(-3, 0.8, r'$-U/2$'); ax.text(2.1, 0.8, r'$+U/2$')

# We expect the width of the peak to be Z*ImΣ(0)
Gamma = -solver_results.Sigma_iw['up_0'][0,0](0).imag
ax.axvspan(-0.5*Z*Gamma, 0.5*Z*Gamma, color='grey', lw=0, alpha=0.25); ax.text(0.05, 0.6, r'$\sim$Z$\Gamma$')
plt.show()

### 🧪 Exercise 4: Analytically continue the Matsubara self-energy

In this exercise, we will analytically continue the Matsubara self-energy onto the real-frequency axis. Due to implementation details, you can directly use the same Python code to analytically continue a self-energy in the same way as a Green's function in TRIQS. 

In [ ]:
n_w = 1000
window = (-10, 10)

mesh = MeshReFreq(window, n_w)
# Step 1: construct Block Green's function on real-frequency mesh
Sigma_w_pade = BlockGf(mesh=mesh, gf_struct=gf_struct)

# Step 2: loop over the blocks and call set_from_pade
n_points = 100
freq_offset = 0.0
for block, g in Sigma_w_pade: g.set_from_pade(solver_results.Sigma_iw[block], n_points=n_points, freq_offset=freq_offset)

# Step 3: plot the spectral function
fig, ax = plt.subplots(1,2,sharex=True, figsize=(7,3))
ax[0].oplot(Sigma_w_pade['up_0']-Sigma_w_pade['up_0'][0,0](0.0).real, mode='R', lw=1, label='Re')
ax[1].oplot(-Sigma_w_pade['up_0'], mode='I', lw=1, label='Im')
ax[0].set_ylim(-5,5); ax[0].set_xlim(-5,5); ax[1].set_ylim(0, ); 
ax[0].set_xlabel('')
ax[0].set_ylabel(r'Re$\Sigma(\omega)$'); ax[1].set_ylabel(r'-Im$\Sigma(\omega)$')
plt.subplots_adjust(wspace=0.5)
plt.show()

In [ ]:
from utils.maxent import Sigma_w_from_maxent

Sigma_w = Sigma_w_from_maxent(solver_results.Sigma_iw)

In [ ]:
fig, ax = plt.subplots(1,2,sharex=True, figsize=(7,3))
# maxent
ax[0].oplot(Sigma_w['up_0']-Sigma_w['up_0'][0,0](0.0).real, mode='R', lw=1, label='MaxEnt')
ax[1].oplot(-Sigma_w['up_0'], mode='I', lw=1, label='MaxEnt')
# pade
ax[0].oplot(Sigma_w_pade['up_0']-Sigma_w_pade['up_0'][0,0](0.0).real, mode='R', lw=1, label='Pade')
ax[1].oplot(-Sigma_w_pade['up_0'], mode='I', lw=1, label='Pade')
ax[0].legend(loc='upper left'); ax[1].legend(loc='upper left')
ax[0].set_ylim(-5,5); ax[0].set_xlim(-2,2); ax[1].set_ylim(0, ); 
ax[0].set_ylabel(r'Re$\Sigma(\omega)$'); ax[1].set_ylabel(r'-Im$\Sigma(\omega)$')
plt.show()

### 🧪 Exercise 5: Estimate Z and the scattering rate from the self-energy

Using the self-energy that you obtained above, estimate the quasiparticle renormalization factor and the scattering rate, which are defined as 

$$ Z = (1 - \partial_{\omega}\text{Re}\Sigma(\omega) |_{\omega\rightarrow0})^{-1}.$$
The scattering rate $\Gamma$ is proportional to $-\mathrm{Im}\Sigma(\omega=0)$. For $Z$, you can fit a linear line to the low-frequency data in the real part of spectrum and use the slope of the line to estimate the derivate.  For the scattering rate, you can fit a polynomial to the low-frequency data in the imaginary part of the self-energy and estimate $-\text{Im}\Sigma(0)$ from this fit.

In [ ]:
x = np.linspace(-0.2, 0.2,10)
y = [Sigma_w['up_0'][0,0](xx).real for xx in x]
dSdw = np.polyfit(x,y,1)[0]
Z = 1/(1 - dSdw)
print(f'     Z  =  {Z}')
print(f'-ImΣ(0) = {Sigma_w['up_0'][0,0](0.0).imag}')

## 📊 Plotting your results

### 🧪 Exercise 6: Plot the momentum-resolved spectral function along the high-symmetry path in the Brillouin zone

### 🧪 Exercise 7: Describe your results
In a few words, describe your spectral function results $A(k, \omega)$. Compare the interacting spectrum to the non-interacting. Can you connect any of these changes to features in the self-energy?

From the momentum-resolved spectral function shown above, we see two distinct features: upper and lower Hubbard bands forming around $\pm U/2$ and a signficant band renormalization due to small quasiparticle renormalization factor, $Z$.